In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import lightgbm as lgb

# 1. Feature Extraction Function
def extract_audio_features(file_path):
    try:
        # Load audio and resample to 16kHz for consistency
        y, sr = librosa.load(file_path, sr=16000, duration=5.0) 
        
        # Extract features
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
        mel = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr).T, axis=0)
        
        # Flatten and combine into a uniform feature vector
        return np.hstack([mfccs, mel, chroma])
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# 2. Dataset Loading Setup
def load_dataset(dataset_dir):
    features = []
    labels = []
    
    # Assuming 'genuine' and 'fake' (or deepfake) are subdirectories
    # Adjust folder names if your directory structure looks slightly different
    categories = {'genuine': 0, 'fake': 1, 'deepfake': 1, 'human': 0}
    
    for folder_name, label in categories.items():
        folder_path = os.path.join(dataset_dir, folder_name)
        if not os.path.exists(folder_path):
            continue
            
        print(f"Loading files from: {folder_path}")
        for file in os.listdir(folder_path):
            if file.endswith(('.wav', '.mp3', '.flac')):
                file_path = os.path.join(folder_path, file)
                feat = extract_audio_features(file_path)
                if feat is not None:
                    features.append(feat)
                    labels.append(label)
                    
    return np.array(features), np.array(labels)

# --- EXECUTION PIPELINE ---
if __name__ == "__main__":
    # TODO: Put your actual extracted directory path here!
    DATASET_PATH = "path/to/the-fake-or-real-dataset/LA_norm/train" 
    
    print("🚀 Starting Feature Extraction (this may take a few minutes)...")
    X, y = load_dataset(DATASET_PATH)
    
    print(f"Data Loaded: {X.shape[0]} samples found.")
    
    # Split Data into Train and Validation
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("🏋️ Training LightGBM Model...")
    # Using class_weight='balanced' to naturally mitigate any dataset imbalances
    model = lgb.LGBMClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    
    # Predict Probabilities for EER calculation
    y_prob = model.predict_proba(X_val)[:, 1]
    
    # 3. Calculate Equal Error Rate (EER) and Optimize Threshold
    fpr, tpr, thresholds = roc_curve(y_val, y_prob, pos_label=1)
    fnr = 1 - tpr
    
    # Find the threshold where FPR == FNR
    eer_threshold = thresholds[np.nanargmin(np.absolute((fpr - fnr)))]
    eer = fpr[np.nanargmin(np.absolute((fpr - fnr)))]
    
    # Apply optimized threshold for final classification metrics
    y_pred = (y_prob >= eer_threshold).astype(int)
    
    # 4. Evaluate Mandatory Performance Metrics
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred)
    
    # Calculate Per-Class Accuracy
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    
    print("\n📊 --- VERIFICATION REPORT ---")
    print(f"Overall Accuracy: {acc*100:.2f}% (Target: >=80%)") [cite: 138, 156]
    print(f"Equal Error Rate (EER): {eer*100:.2f}% (Target: <=12%)") [cite: 142, 156]
    print(f"F1 Score: {f1*100:.2f}% (Target: >=80%)") [cite: 148]
    print(f"Genuine Class Accuracy: {per_class_acc[0]*100:.2f}% (Target: >=75%)") [cite: 152]
    print(f"Deepfake Class Accuracy: {per_class_acc[1]*100:.2f}% (Target: >=75%)") [cite: 152]
    print("\nConfusion Matrix:") [cite: 145]
    print(cm)
    
    # Save the artifact models and threshold configurations
    payload = {
        'model': model,
        'eer_threshold': eer_threshold
    }
    joblib.dump(payload, 'deepfake_detector_model.pkl')
    print("\n💾 Model artifacts saved successfully as 'deepfake_detector_model.pkl'!")